# Alice/Bob без барьера после каждого токена
Два независимых цикла генерации. AsyncLLM сам собирает готовые запросы в batch.
Синхронизация остаётся только при публикации: нельзя очищать блок, на который ссылается queued request.
`PublicationGate` допускает параллельные forward, но делает flush/append/clear/prefill эксклюзивными.
Kernel **AsyncLLM (uv)** → **Run All**.


In [1]:
import asyncio
import gc
import os
import sys
import time
from contextlib import aclosing
from pathlib import Path

from IPython.display import Markdown, display

# Start Jupyter in the repository root or scripts/parallel_thinking.
ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="1",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import PublicationGate, isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay

TEACHER = "models/Qwen3-4B-Thinking-2507"
STUDENT = "models/Qwen3-1.7B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
async def solve_no_barrier(model, *, max_steps=1024, prompt_ids=None, bob_delay=0.0):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)
        # The original QwQ prompt also matches the Thinking-2507 template exactly.
        fmt = MathFormatting(tokenizer, WORKERS)
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )
        # Graph profiles admit two requests, so initialize the long prompt first.
        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]
        # Leave the last prefix token pending, so round 1 is a joint decode.
        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>", "</think>")
            if t in tokenizer.vocab
        ]
        # EOS remains a stop condition, even if a model uses endoftext as EOS.
        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        gate = PublicationGate()
        stopped = False
        progress = []

        async def generate_agent(i):
            nonlocal stopped
            ctx = contexts[i]
            for local_step in range(1, max_steps + 1):
                if stopped:
                    break
                # Optional delay demonstrates that Alice does not wait for Bob.
                if i == 1 and bob_delay and local_step % 32 == 0:
                    await asyncio.sleep(bob_delay)
                async with gate.forward():
                    token = await one_token(llm, ctx, forbidden)
                    current[i].append(token)
                    generated[i] += 1
                    progress.append(tuple(generated))
                if token == eos:
                    stopped = True
                    break
                if fmt.is_end_of_step(current[i]):
                    # Only publication needs exclusive access. Pause submissions,
                    # settle existing forwards, then flush/append/clear/prefill.
                    async with gate.publish():
                        await llm.forward(cache_view=ctx, return_logits=False)
                        block = worker_blocks[i]
                        assert block.token_ids == current[i]
                        await llm.append_block(common, block)
                        history.extend(current[i])
                        assert common.token_ids == initial_ids + history
                        events.append(
                            {
                                "worker": WORKERS[i],
                                "step": step_indices[i],
                                "tokens": len(current[i]),
                                "generated": tuple(generated),
                            }
                        )
                        await llm.free_block(block)
                        step_indices[i] += 1
                        current[i] = encode(fmt.get_step_prefix(WORKERS[i], step_indices[i]))
                        await llm.forward(current[i][:-1], cache_view=ctx, return_logits=False)
                        ctx.next_input_id = current[i][-1]
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]

        # One join at the end of two independent loops, not at every token.
        await together(generate_agent(0), generate_agent(1))
        iteration = max(generated)

        # Flush both final sampled tokens before the answer continuation reads them.
        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["progress"] = progress
        result["max_token_gap"] = max(abs(a - b) for a, b in progress)
        result["example"] = recorded_example(llm)
        return result
    finally:
        # No active streams or queued requests remain here, including on errors.
        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

In [3]:
normal = isolated(solve_no_barrier, TEACHER)
print("Answer:", normal["answer"], "Published blocks:", normal["completed_steps"])
print("Generated:", normal["generated_tokens"], "Maximum token gap:", normal["max_token_gap"])
display(Markdown(normal["transcript"] + "\n\n" + normal["answer_text"]))

Answer: 294.0 Published blocks: 8
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 1


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll compute the distances between the points to find the edge length of the cube. Let's start with PQ, QR, and RP.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three vertices. First, I should find the distances between P, Q, and R. Let's compute PQ, QR, and RP.

**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)
**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Compute the edge length. Since all three distances are equal to 7*sqrt(2), which is the face diagonal of the cube (s*sqrt(2)), then the edge length s = 7. Wait, is that right? Let me think. If the face diagonal is s*sqrt(2), then s = (7*sqrt(2)) / sqrt(2) = 7. So edge length is 7.

**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Compute the edge length. Since all three distances are equal to 7*sqrt(2), which is the face diagonal of the cube (s*sqrt(2)), then the edge length s = 7. Wait, is that right? Let me think. If the face diagonal is s*sqrt(2), then s = (7*sqrt(2)) / sqrt(2) = 7. So edge length is 7.

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.



### Work in progress (others)

**Alice [5]:** 5. So, if the distance between each pair is a face diagonal, then the edge length s = 7. Then surface area is 6*s^2 = 6*49 = 294<...>

**Bob [5]:** 4. Compute the edge length. Since all three distances are equal to 7*sqrt(2), which is the face diagonal<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}

## Проверка независимого темпа
Каждые 32 токена искусственно задерживаем Bob на 50 мс. Alice может продолжать работу,
пока Bob спит. Разрыв счётчиков больше единицы демонстрирует отсутствие пошагового барьера.
Содержимое и порядок публикаций могут измениться: при другом расписании модель видит другой контекст.


In [4]:
delayed = isolated(solve_no_barrier, TEACHER, bob_delay=0.05)
print("Answer:", delayed["answer"], "Published blocks:", delayed["completed_steps"])
print("Generated:", delayed["generated_tokens"], "Maximum token gap:", delayed["max_token_gap"])
print("Same trajectory:", delayed["history_ids"] == normal["history_ids"])
assert delayed["max_token_gap"] > 1
# Check that both independent runs produce valid replay plans for training.
for result in (normal, delayed):
    example = result["example"]
    replay = Replay.from_trace(
        example["trace"], query_chunk_size=64, attention_checkpointing=True, cache_plans=True
    )
    print(
        "Trace rows:",
        example["trace"].num_tokens,
        "Targets:",
        sum(t != -100 for t in example["targets"]),
    )
display(Markdown(delayed["transcript"] + "\n\n" + delayed["answer_text"]))

Answer: 294.0 Published blocks: 8
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 64
Same trajectory: False
Trace rows: 3243 Targets: 2052


Trace rows: 3243 Targets: 2052


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll compute the distances between the points to find the edge length of the cube. Let's start with PQ, QR, and RP.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three vertices. First, I should find the distances between P, Q, and R. Let's calculate PQ, QR, and RP.

**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means P, Q, R are all equidistant from each other. In a cube, if three vertices are equidistant, they must form an equilateral triangle. But wait, in a cube, the only equilateral triangles formed by vertices are those with side length equal to the face diagonal (which is edge length * sqrt(2)). So if the side length of the triangle is 7*sqrt(2), then the edge length of the cube is 7.

**Alice [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)
**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Okay, so all three distances are equal to 7*sqrt(2). That's interesting. In a cube, the space diagonal is longer than the face diagonal, which is longer than the edge length. But here, all three distances are equal. Wait, in a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal. So the side length of the triangle is the face diagonal, which is edge length * sqrt(2). So if the face diagonal is 7*sqrt(2), then edge length is 7.

**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Okay, so all three distances are equal to 7*sqrt(2). That's interesting. In a cube, the space diagonal is longer than the face diagonal, which is longer than the edge length. But here, all three distances are equal. Wait, in a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal. So the side length of the triangle is the face diagonal, which is edge length * sqrt(2). So if the face diagonal is 7*sqrt(2), then edge length is 7.

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [6]:** So all three distances are equal to 7*sqrt(2). That means P, Q, R form an equilateral triangle with side length 7*sqrt(2). In a cube, the only equilateral triangles formed by vertices are those where each side is a face diagonal. So the face diagonal is 7*sqrt(2), which means the edge length of the cube is 7 (since face diagonal = edge * sqrt(2) => edge = face diagonal / sqrt(2) = 7*sqrt(2)/sqrt(2) = 7).

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [6]:** So all three distances are equal to 7*sqrt(2). That means P, Q, R form an equilateral triangle with side length 7*sqrt(2). In a cube, the only equilateral triangles formed by vertices are those where each side is a face diagonal. So the face diagonal is 7*sqrt(2), which means the edge length of the cube is 7 (since face diagonal = edge * sqrt(2) => edge = face diagonal / sqrt(2) = 7*sqrt(2)/sqrt(2) = 7).

**Bob [5]:** 4. Now, I need to confirm if these three points are indeed vertices of a face diagonal triangle. Let's check the vectors. Let's compute the vectors PQ, QR, RP.



### Work in progress (others)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** Okay, so<...>

**Bob [6]:** 4. Let's compute<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}